# STRAT-06 Comparative Analysis

**Period:** 2017-08-17 → 2026-05-06 (3,185 daily bars, 8.5 years)  
**Capital:** €52,500 per strategy (€500/month × 105 months)  
**Strategies:** Buy-and-Hold · DCA Benchmark · STRAT-06  

This notebook loads the output of `scripts/run_strat06_comparative_backtest.py` and
produces visualisations and tables for the historical validation report.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

# ── Load backtest output ──────────────────────────────────────────────────────
REPORT_DIR = Path("../data/reports/strat06_comparative_20260511_193601")

with open(REPORT_DIR / "comparative_summary.json") as f:
    summary = json.load(f)

ec_bah    = pd.read_csv(REPORT_DIR / "equity_curve_bah.csv",    parse_dates=["timestamp_utc"])
ec_dca    = pd.read_csv(REPORT_DIR / "equity_curve_dca.csv",    parse_dates=["timestamp_utc"])
ec_strat  = pd.read_csv(REPORT_DIR / "equity_curve_strat06.csv", parse_dates=["timestamp_utc"])

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

EUR   = FuncFormatter(lambda x, _: f"€{x:,.0f}")
PCT   = FuncFormatter(lambda x, _: f"{x*100:.0f}%")
COLORS = {"bah": "#2196F3", "dca": "#4CAF50", "strat06": "#FF9800"}

print("Data loaded.",
      f"Period: {summary['backtest_period']['start']} → {summary['backtest_period']['end']}",
      f"| {summary['backtest_period']['total_bars']} bars")

## 1 · Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})

ax_eq, ax_ratio = axes

# ── Top: equity curves (log scale) ───────────────────────────────────────────
ax_eq.semilogy(ec_bah["timestamp_utc"],   ec_bah["total_value_eur"],
               color=COLORS["bah"],    lw=1.6, label="Buy-and-Hold")
ax_eq.semilogy(ec_dca["timestamp_utc"],   ec_dca["total_value_eur"],
               color=COLORS["dca"],    lw=1.6, label="DCA Benchmark")
ax_eq.semilogy(ec_strat["timestamp_utc"], ec_strat["total_value_eur"],
               color=COLORS["strat06"], lw=1.6, label="STRAT-06")

ax_eq.axhline(52_500, color="gray", lw=0.8, ls="--", label="Capital injected (€52,500)")
ax_eq.set_ylabel("Portfolio value (EUR, log)")
ax_eq.legend(loc="upper left")
ax_eq.yaxis.set_major_formatter(EUR)
ax_eq.set_title("STRAT-06 vs Benchmarks — Equity Curves (2017–2026)", fontweight="bold")

# ── Bottom: STRAT-06 / DCA ratio ─────────────────────────────────────────────
# Align on common index
merged = ec_strat[["timestamp_utc", "total_value_eur"]].merge(
    ec_dca[["timestamp_utc", "total_value_eur"]], on="timestamp_utc",
    suffixes=("_s06", "_dca")
)
ratio = merged["total_value_eur_s06"] / merged["total_value_eur_dca"].replace(0, np.nan)
ax_ratio.plot(merged["timestamp_utc"], ratio, color=COLORS["strat06"], lw=1.2)
ax_ratio.axhline(1.0, color="gray", lw=0.8, ls="--")
ax_ratio.set_ylabel("STRAT-06 / DCA ratio")
ax_ratio.set_xlabel("Date")
ax_ratio.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout()
plt.show()

## 2 · Metrics Comparison Table

In [ ]:
strats = summary["strategies"]

rows = []
for s in strats:
    rows.append({
        "Strategy":         s["strategy"].replace("_", " ").title(),
        "Invested (€)":     f"€{s['total_invested_eur']:>10,.0f}",
        "Final (€)":        f"€{s['final_value_eur']:>10,.0f}",
        "BTC":              f"{s['btc_accumulated']:.4f}",
        "Cash idle (€)":    f"€{s['cash_remaining_eur']:>8,.0f}",
        "Return %":         f"{s['total_return_pct']:.1f}%",
        "CAGR %":           f"{s['cagr_pct']:.1f}%" if s["cagr_pct"] else "N/A",
        "MaxDD %":          f"{s['max_drawdown_pct']:.1f}%",
        "Sharpe":           f"{s['sharpe_ratio']:.2f}" if s["sharpe_ratio"] else "N/A",
        "Sortino":          f"{s['sortino_ratio']:.2f}" if s["sortino_ratio"] else "N/A",
        "Fees (€)":         f"€{s['total_fees_paid_eur']:.0f}",
        "Time in market":   f"{s['time_in_market_pct']:.1f}%",
    })

df_metrics = pd.DataFrame(rows).set_index("Strategy")

# Add excess-return row for STRAT-06 vs benchmarks
er = summary["excess_returns"]
print("Comparative metrics:")
display(df_metrics)
print(f"\nSTRAT-06 excess return vs Buy-and-Hold : {er['strat06_vs_bah_pct']:+.1f} pp")
print(f"STRAT-06 excess return vs DCA Benchmark: {er['strat06_vs_dca_pct']:+.1f} pp")

## 3 · Pool Attribution — Where Did STRAT-06 Deploy Its Capital?

In [ ]:
# Rebuild pool cash series from equity curve columns
pool_cols = [c for c in ec_strat.columns if c.startswith("cash_")]
print("Pool columns found:", pool_cols)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

labels = {"cash_strat06_baseline_eur": "BASELINE",
           "cash_strat06_buffer_eur":  "BUFFER",
           "cash_strat06_reserve_eur": "RESERVE"}
pool_colors = {"cash_strat06_baseline_eur": "#FF9800",
               "cash_strat06_buffer_eur":  "#9C27B0",
               "cash_strat06_reserve_eur": "#F44336"}

for ax, col in zip(axes, ["cash_strat06_baseline_eur",
                            "cash_strat06_buffer_eur",
                            "cash_strat06_reserve_eur"]):
    ax.fill_between(ec_strat["timestamp_utc"], ec_strat[col],
                    alpha=0.4, color=pool_colors[col])
    ax.plot(ec_strat["timestamp_utc"], ec_strat[col],
            color=pool_colors[col], lw=1.2)
    ax.set_ylabel(labels[col])
    ax.yaxis.set_major_formatter(EUR)

axes[0].set_title("STRAT-06 Pool Cash Balances Over Time", fontweight="bold")
axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout()
plt.show()

# ── Summary stats ─────────────────────────────────────────────────────────────
TOTAL_INJECTED = 52_500.0
print("\nCapital accounting (STRAT-06):")
print(f"  Total injected             : €{TOTAL_INJECTED:>10,.2f}")
print(f"  BTC cost basis (deployed)  : €{TOTAL_INJECTED - 5_105.35:>10,.2f}  (est.)")
print(f"  Residual cash              : €{5_105.35:>10,.2f}  (9.7%)")
print(f"    BASELINE pool cash       : €{137.64:>10,.2f}")
print(f"    BUFFER  pool cash        : €{3_467.71:>10,.2f}")
print(f"    RESERVE pool cash (cap)  : €{1_500.00:>10,.2f}")

## 4 · Reserve Deployments

In [ ]:
deployments = summary["strat06_reserve_deployments"]
df_dep = pd.DataFrame(deployments)
df_dep["timestamp_utc"] = pd.to_datetime(df_dep["timestamp_utc"])
df_dep["date"] = df_dep["timestamp_utc"].dt.date

# ── Table ─────────────────────────────────────────────────────────────────────
print(f"{len(df_dep)} reserve tranche deployments\n")
display(df_dep[["date", "reserve_before_eur", "amount_deployed_eur", "reserve_after_eur"]]
        .rename(columns={"reserve_before_eur": "Reserve before",
                          "amount_deployed_eur": "Deployed (€)",
                          "reserve_after_eur": "Reserve after"}))

# ── Plot: reserve cash over time + deployment markers ────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(ec_strat["timestamp_utc"], ec_strat["cash_strat06_reserve_eur"],
                alpha=0.3, color="#F44336")
ax.plot(ec_strat["timestamp_utc"], ec_strat["cash_strat06_reserve_eur"],
        color="#F44336", lw=1.2, label="RESERVE cash")

for _, row in df_dep.iterrows():
    ax.axvline(row["timestamp_utc"], color="#B71C1C", lw=1.5, ls="--", alpha=0.7)
    ax.annotate(f"€{row['amount_deployed_eur']:.0f}",
                xy=(row["timestamp_utc"], row["reserve_after_eur"]),
                xytext=(0, 8), textcoords="offset points",
                fontsize=8, ha="center", color="#B71C1C")

ax.axhline(1_500, color="gray", lw=0.8, ls=":", label="Reserve cap (€1,500)")
ax.set_ylabel("Reserve cash (EUR)")
ax.set_xlabel("Date")
ax.set_title("STRAT-06 Reserve Pool — Balance and Deployments", fontweight="bold")
ax.legend()
ax.yaxis.set_major_formatter(EUR)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 5 · BTC Cost Basis Comparison

In [ ]:
# Compute running average cost basis = cumulative EUR spent / cumulative BTC bought
# We approximate from equity curve: BTC value = total_value_eur - cash_pools

# For STRAT-06: extract total_cash per bar
ec_strat["total_cash_eur"] = (
    ec_strat["cash_strat06_baseline_eur"]
    + ec_strat["cash_strat06_buffer_eur"]
    + ec_strat["cash_strat06_reserve_eur"]
)

# Approximate: we know the final avg cost basis from summary
strat06_summary = next(s for s in strats if s["strategy"] == "strat06")
dca_summary     = next(s for s in strats if s["strategy"] == "dca_benchmark")
bah_summary     = next(s for s in strats if s["strategy"] == "buy_and_hold")

# Cost basis from summary (only STRAT-06 has avg_cost_basis_eur_per_btc)
strat06_avg_cost = strat06_summary.get("avg_cost_basis_eur_per_btc")

# For BaH: lump-sum on first bar
bah_avg_cost = bah_summary["total_invested_eur"] / bah_summary["btc_accumulated"]
# For DCA: total invested / total btc
dca_avg_cost = dca_summary["total_invested_eur"] / dca_summary["btc_accumulated"]

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
strategies = ["Buy-and-Hold", "DCA Benchmark", "STRAT-06"]
costs = [bah_avg_cost, dca_avg_cost, strat06_avg_cost]
colors_list = [COLORS["bah"], COLORS["dca"], COLORS["strat06"]]
bars = ax.bar(strategies, costs, color=colors_list, alpha=0.85, edgecolor="white", lw=1.5)

for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
            f"€{cost:,.0f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Avg cost basis (EUR/BTC)")
ax.set_title("Average BTC Cost Basis per Strategy", fontweight="bold")
ax.yaxis.set_major_formatter(EUR)
plt.tight_layout()
plt.show()

print(f"Buy-and-Hold  avg cost: €{bah_avg_cost:>10,.2f}/BTC  (lump-sum 2017, lowest entry)")
print(f"DCA Benchmark avg cost: €{dca_avg_cost:>10,.2f}/BTC  (weekly, uniform)")
if strat06_avg_cost:
    print(f"STRAT-06      avg cost: €{strat06_avg_cost:>10,.2f}/BTC  (modulated + reserve tranches)")

## 6 · Drawdown Comparison

In [ ]:
def running_drawdown(equity_series: pd.Series) -> pd.Series:
    """Period-to-period drawdown from running peak. Returns negative fractions."""
    nonzero = equity_series.replace(0, np.nan).dropna()
    if nonzero.empty:
        return pd.Series(0.0, index=equity_series.index)
    peak = equity_series.expanding().max()
    peak = peak.where(peak > 0)
    dd = (equity_series - peak) / peak
    return dd.fillna(0)

dd_bah   = running_drawdown(ec_bah["total_value_eur"])
dd_dca   = running_drawdown(ec_dca["total_value_eur"])
dd_strat = running_drawdown(ec_strat["total_value_eur"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(ec_bah["timestamp_utc"],   dd_bah   * 100, 0,
                alpha=0.25, color=COLORS["bah"],    label="Buy-and-Hold")
ax.fill_between(ec_dca["timestamp_utc"],   dd_dca   * 100, 0,
                alpha=0.25, color=COLORS["dca"],    label="DCA Benchmark")
ax.fill_between(ec_strat["timestamp_utc"], dd_strat * 100, 0,
                alpha=0.35, color=COLORS["strat06"], label="STRAT-06")

ax.plot(ec_bah["timestamp_utc"],   dd_bah   * 100, color=COLORS["bah"],    lw=0.8)
ax.plot(ec_dca["timestamp_utc"],   dd_dca   * 100, color=COLORS["dca"],    lw=0.8)
ax.plot(ec_strat["timestamp_utc"], dd_strat * 100, color=COLORS["strat06"], lw=1.0)

ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.set_title("Drawdown Comparison — All Three Strategies", fontweight="bold")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

# ── Max drawdown summary ──────────────────────────────────────────────────────
print("Max drawdown:")
for label, dd_series in [("Buy-and-Hold", dd_bah),
                          ("DCA Benchmark", dd_dca),
                          ("STRAT-06",      dd_strat)]:
    print(f"  {label:<20}: {dd_series.min()*100:>6.1f}%")

## 7 · Risk-Adjusted Performance

In [ ]:
# Radar / spider chart: Sharpe, Sortino, MaxDD (inverted), Return/MaxDD
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Scatter: Return% vs MaxDD% ────────────────────────────────────────────────
ax = axes[0]
for s in strats:
    label = s["strategy"].replace("_", "-").upper()
    color = COLORS.get(s["strategy"].replace("_benchmark", "").replace("buy_and_hold", "bah"),
                       COLORS["strat06"])
    ax.scatter(abs(s["max_drawdown_pct"]), s["total_return_pct"],
               s=200, color=color, zorder=5, edgecolors="white", lw=1.5)
    ax.annotate(label, (abs(s["max_drawdown_pct"]), s["total_return_pct"]),
                xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.set_xlabel("Max Drawdown (abs %)")
ax.set_ylabel("Total Return (%)")
ax.set_title("Return vs Max Drawdown", fontweight="bold")

# ── Bar chart: Sharpe / Sortino ───────────────────────────────────────────────
ax2 = axes[1]
names  = [s["strategy"].replace("_", "\n").title() for s in strats]
sharpe  = [s["sharpe_ratio"] or 0 for s in strats]
sortino = [s["sortino_ratio"] or 0 for s in strats]

x = np.arange(len(names))
w = 0.35
bars1 = ax2.bar(x - w/2, sharpe,  w, label="Sharpe",  color="#607D8B", alpha=0.85)
bars2 = ax2.bar(x + w/2, sortino, w, label="Sortino", color="#795548", alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(names)
ax2.set_ylabel("Ratio")
ax2.set_title("Risk-Adjusted Metrics", fontweight="bold")
ax2.legend()

for bar in list(bars1) + list(bars2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
plt.show()

## 8 · Conclusions

### What the historical backtest shows

| Dimension | STRAT-06 vs DCA Benchmark |
|---|---|
| Final value | €273,105 vs €285,998 (−4.5%) |
| Total return | 420.2% vs 444.8% (−24.6 pp) |
| Max drawdown | −73.9% vs −74.8% (+0.9 pp **better**) |
| Sharpe ratio | 1.34 vs 1.32 (+0.02 **better**) |
| Sortino ratio | 2.17 vs 2.07 (+0.10 **better**) |
| Idle cash | €5,105 (9.7%) vs €424 (0.8%) |

### Interpretation

**STRAT-06 achieves marginally better risk-adjusted performance than DCA** (higher Sharpe,
higher Sortino, shallower max drawdown) at the cost of ~4.5% lower final portfolio value.
The performance gap is largely explained by two structural features:

1. **RESERVE design cost:** The reserve pool caps at €1,500 and holds dry powder for deep-value
   entries. Its 8 deployments (4 in 2018, 4 in 2022) bought BTC at historically low prices,
   but the remaining €1,500 and €3,468 BUFFER residual represent capital that was deliberately
   not deployed — a conscious trade-off.

2. **Modulation timing:** The drawdown-modulated buying (78.6% of baseline buys had multiplier > 1×)
   correctly directed capital toward bear markets, but the modulation effect is limited by the
   available BASELINE cash on each Monday (mean multiplier 1.81×, max theoretical 2.5×).

### Design refinement applied

The `max_concentration_pct` guard was removed (set to 1.0) per
`docs/reports/STRAT_06_DESIGN_REFINEMENT_001.md`. Before the fix, 363 of 455 eligible
baseline Mondays were blocked, leaving 69% of capital idle — a pathological outcome.

### What is NOT shown here (honest limitations)

- **Selection bias:** We are comparing against BTC-denominated benchmarks over a period where
  BTC was the best-performing asset class globally. STRAT-06 underperforming pure DCA on BTC
  is not necessarily a failure — it may reflect a capital-conservation design that would shine
  in sideways or declining markets.
- **No out-of-sample period:** All parameters (tranche thresholds, multiplier table,
  persistence days) were designed prior to this backtest, but the strategy has only been
  validated on the same period used for design. Future validation on post-2026 data is required.
- **Tax and execution costs** are simplified (0.1% fee flat). Real-world slippage on small weekly
  orders may be lower than the model assumes.